In [ ]:
import importlib

import dotenv

dotenv.load_dotenv()

In [ ]:
import pyine.prompts.code_execution
import pyine.utils.code.execution
import pyine.utils.portability

importlib.reload(pyine.prompts.code_execution)
importlib.reload(pyine.utils.code.execution)
importlib.reload(pyine.utils.portability)

In [ ]:
import langchain_core.globals

langchain_core.globals.set_debug(True)
# langchain_core.globals.set_verbose(True)

In [ ]:
# BUG-FREE CODE SNIPPET (USEFUL DURING TRAINING)

example_snippet = """\
def calculate_area(length: float, width: float) -> float:
    '''Returns the area of the rectangle specified via length and width.

    Specifically: returns area = length * width.
    '''
    area = length * width
    print(f"The area of the rectangle is: {area:.2f} square units")
    return area

length = float(input("Enter the length: "))
width = float(input("Enter the width: "))
calculate_area(length, width)
"""
example_input_args = """\
5.0
3.0
"""

In [ ]:
# BUGGY CODE SNIPPET (USEFUL DURING EVALUATIONS)
# (see the computation of the area itself: it's a fake-fix-me instead of actual code)

example_snippet = """\
def calculate_area(length: float, width: float) -> float:
    '''Returns the area of the rectangle specified via length and width.

    Specifically: returns area = length * width.
    '''
    area = todo # TODO: not implemented, use `length * width` here.
    print(f"The area of the rectangle is: {area:.2f} square units")
    return area

width = float(input("Enter the width: "))
length = float(input("Enter the length: "))
calculate_area(length, width)
"""
example_input_args = """\
5.0
3.0
"""

In [ ]:
# 'ground truth' code tracing demo (using the python interpreter directly)
trace_result = pyine.utils.code.execution.execute_and_trace_code(
    example_snippet,
    example_input_args,
    trace_only_inside_code_string=True,
    max_events_per_line=100,
)

print("\nTraced code string:")
pyine.utils.portability.print_code_with_numbered_lines(example_snippet, 1)
print("\nTraced steps:")
for traced_step_idx, traced_step in enumerate(trace_result.traced_steps):
    if traced_step is None:
        print(f"\tstep#{traced_step_idx:04d}:\t(out-of-context execution)")
    else:
        print(f"\tstep#{traced_step_idx:04d}:\t{traced_step}")
if trace_result.return_value is not None:
    print(f"\nCaptured return value:\n\t{trace_result.return_value}")
if trace_result.exception is not None:
    print(f"\nCaptured exception:\n\t{trace_result.exception}")
if trace_result.stdout:
    print(f"\nCaptured output:\n\t{trace_result.stdout}")
if trace_result.stderr:
    print(f"\nCaptured error:\n\t{trace_result.stderr}")

In [ ]:
# display prompt used for all models in subsequent cells:
example_prompt = pyine.prompts.code_execution.code_execution_prompt.format(
    code=example_snippet,
    input_args=example_input_args,
    examples=pyine.prompts.code_execution._code_execution_examples_str,
)
print(example_prompt)

In [ ]:
# naive system 1 code execution demo (using deepseek-chat)
code_exec_chain = pyine.prompts.code_execution.get_chain(
    provider="deepseek",
    model="deepseek-chat",
    temperature=0.0,
)
response = code_exec_chain.invoke({"code": example_snippet, "input_args": example_input_args})
print(f"\n\nFinal deepseek-chat prediction:\n{response.content}")

In [ ]:
# naive system 1 code execution demo (using gpt4o)
code_exec_chain = pyine.prompts.code_execution.get_chain(
    provider="openai",
    model="gpt-4o",
    temperature=0.0,
)
response = code_exec_chain.invoke({"code": example_snippet, "input_args": example_input_args})
print(f"\n\nFinal gpt-4o prediction:\n{response.content}")

In [ ]:
# weakest system 2 code execution demo (using deepseek-r1)
code_exec_chain = pyine.prompts.code_execution.get_chain(
    provider="deepseek",
    model="deepseek-reasoner",
    temperature=0.0,
    max_tokens=10_000,
)
response = code_exec_chain.invoke({"code": example_snippet, "input_args": example_input_args})
print(f"\nReasoning:\n{response.additional_kwargs['reasoning_content']}")
print(f"\nFinal prediction:\n{response.content}")

In [ ]:
# weakest system 2 code execution demo (using o3)
code_exec_chain = pyine.prompts.code_execution.get_chain(
    provider="openai",
    model="o3",
    temperature=0.0,
    max_tokens=10_000,
)
response = code_exec_chain.invoke({"code": example_snippet, "input_args": example_input_args})
print(f"\nReasoning:\n{response.additional_kwargs['reasoning_content']}")
print(f"\nFinal prediction:\n{response.content}")